# Find duplicate files

In [ ]:
#@title Connect Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#@title Find
folder = "/content/drive/MyDrive" #@param {type:"string"}

# Accept paths as Drive shows them (Temp\video.mp4) or as full /content paths.
folder = folder.strip().strip('"').replace("\\", "/")
if folder and not folder.startswith("/"):
    folder = "/content/drive/MyDrive/" + folder.removeprefix("MyDrive/").removeprefix("My Drive/")

import os, hashlib
def human(n):
    for unit in ["B", "KB", "MB", "GB", "TB"]:
        if n < 1024 or unit == "TB":
            return f"{n:.1f} {unit}"
        n /= 1024

root = folder.rstrip("/")
if not os.path.isdir(root):
    raise SystemExit(f"Not a folder: {root}")
by_size = {}
for dirpath, _, names in os.walk(root):
    for name in names:
        p = os.path.join(dirpath, name)
        try:
            by_size.setdefault(os.path.getsize(p), []).append(p)
        except OSError:
            pass

def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

# Only same-size files can be equal, so hash just those.
groups = {}
for size, paths in by_size.items():
    if size and len(paths) > 1:
        for p in paths:
            groups.setdefault((size, sha256(p)), []).append(p)
dups = sorted(((s, ps) for (s, _), ps in groups.items() if len(ps) > 1), reverse=True)
for size, paths in dups:
    print(human(size))
    for p in sorted(paths):
        print("  ", os.path.relpath(p, root))
print(f"\n{len(dups)} groups, {human(sum(s * (len(ps) - 1) for s, ps in dups))} reclaimable")
